### Cleanup frequencies from cleanup logs

In [7]:
from dataclasses import dataclass

import pandas as pd
from lexical_benchmark.datasets import childes
from lexical_benchmark.utils import timed_status
dataset = childes.CHILDESDataset()


@dataclass
class Stats:
    rejected_word_count: int
    rejected_type_count: int
    pre_word_count: int
    pre_type_count: int


with timed_status(status="Extracting WF from dataset...", complete_status="Finished Extracting WF."):
    cleanup_stats_dct = {}
    for lang_accent in dataset.accents:
        for speech_type in dataset.speech_types:
            rjdf = pd.read_csv(dataset.wf.rejected(lang_accent, speech_type))
            predf = pd.read_csv(dataset.wf.processed(lang_accent, speech_type))
            cleanup_stats_dct[f"{lang_accent}/{speech_type}"] = Stats(
                rejected_word_count=rjdf["freq"].sum(),
                rejected_type_count=len(rjdf["word"]),
                pre_word_count=predf["freq"].sum(),
                pre_type_count=len(predf["word"]),
            )
    cleanup_stats = pd.DataFrame([
        {
            "section": key,
            "Token Rejection Rate": st.rejected_word_count / st.pre_word_count,
            "Type Rejection Rate": st.rejected_type_count / st.pre_type_count,
        } for key, st in cleanup_stats_dct.items()
    ])

Output()

Finished Extracting WF. (Total time: 1 seconds)

# Block Average Rejection Rates

To be able to compare those stats with STELA and other datasets of different size we do a block-avergage computation.

In [8]:
import pandas as pd
from lexical_benchmark.utils import timed_status
from lexical_benchmark.stats import block_average2
from lexical_benchmark.datasets import childes, utils as dataset_utils

childes_extras_id = "9a30b8dad7abe369b2402b989f07e28b"
dataset = childes.CHILDESDataset()
en_dict = dataset_utils.DictionairyCleaner(lang="EN", childes_extra_id=childes_extras_id)
CHUNK_SPLIT = 16_000


def word_clean_fn(word: str) -> bool:
    """Check if a word is in dict"""
    global en_dict
    return en_dict.check(word)


with timed_status(status="Computing WRJ...", complete_status="Finished WRJ"):
    block_rj_rates = {}
    for lang_accent in dataset.accents:
        for speech_type in dataset.speech_types:
            words = []
            for item in dataset.iter_accent(lang_accent):
                words.extend(item.preprocess_item(speech_type).processed.read_tokenized())
            chunk_list = block_average2.chunk_splitter(
                words, chunk_size=CHUNK_SPLIT
            )
            word_chunk_stats = block_average2.clean_chunk_list(
                chunks=chunk_list, filter_fn=word_clean_fn
            )
            block_rj_rates[f"{lang_accent}/{speech_type}"] = word_chunk_stats

chunk_avg_rj_rates = pd.DataFrame([
    {
        "section": label,
        "Token Rejection Rate": cs.rejection_rate,
        "Type Rejection Rate": cs.unique_rejection_rate,
    }
    for label, cs in block_rj_rates.items()
])
!date

Output()

Finished WRJ (Total time: 29 seconds)

Mon Dec 16 04:34:53 PM CET 2024


In [9]:
from IPython.display import display, display_html, HTML

cleanup_stats_st = cleanup_stats.style.format({
    "Token Rejection Rate": "{:.2%}",
    "Type Rejection Rate": "{:.2%}",
})
chunk_avg_rj_rates_st = chunk_avg_rj_rates.style.format({
    "Token Rejection Rate": "{:.2%}",
    "Type Rejection Rate": "{:.2%}",
})
display(HTML("<h3>Clean-up stats</h3>"))
display(cleanup_stats_st)
display(HTML("<h3>Block-AVG clean-up stats (16k blocks)</h3>"))
display(chunk_avg_rj_rates_st)
!date

,section,Token Rejection Rate,Type Rejection Rate
0,Eng-NA/child,1.35%,38.78%
1,Eng-NA/adult,0.48%,18.71%
2,Eng-UK/child,0.62%,26.05%
3,Eng-UK/adult,0.29%,13.73%


,section,Token Rejection Rate,Type Rejection Rate
0,Eng-NA/child,1.05%,4.79%
1,Eng-NA/adult,0.46%,2.09%
2,Eng-UK/child,0.39%,2.00%
3,Eng-UK/adult,0.26%,1.28%


Mon Dec 16 04:36:15 PM CET 2024
